# Synthetic Payments Data Generation for Federated Payments Anomaly Detection

## Introduction

As with any Machine Learning experiment, training and evaluation of models requires good quality datasets.
This is even more pertinent with **Federated Learning (FL)**, where the premise is to train a better model
than individual participant models by combining knowledge from data with different distributions — *without*
sharing the underlying data.

Combined with the lack of good-quality data from payment systems, or customisable dataset generators for
this domain, this creates a fundamental problem.  We solve it here by building a **purely-synthetic data
generation tool** capable of:

1. Generating arbitrary numbers of training and validation datasets.
2. Controlling distributions per-site for repeatability and explainability.
3. Customisable features and attributes via a YAML configuration file.
4. Inducing **rule-based anomalies** with controllable overlap, class sizes, and multi-dimensional scenarios.

### Architecture Overview

The tool is organised into a **library** (`data_generation/`) and a thin orchestration layer.
The library provides:

| Module | Purpose |
|--------|---------|
| `static_data/` | Country-currency mappings, exchange rates, reference constants |
| `rng/` | Seedable RNG wrappers (uniform, normal, log-normal, gamma, random-choice) |
| `synthetic_data_provider/` | Provider classes that pair Faker or an RNG with a uniform `provide()` API |
| `dataset_attribute/` | `PaymentDatasetAttribute` / `PaymentDatasetAttributeGroup` descriptors |
| `attributes.py` | Dependency graph definitions mapping attributes → prerequisite columns |
| `dataset.py` | Generic `topological_sort()` and `generate()` functions |

This notebook walks through each layer step-by-step, culminating in a full dataset generation
with anomaly injection.

### Caveats

- Distributions are hypothetical — real payment distributions are unavailable due to regulations.


## Setup & Configuration

We import our `data_generation` library modules alongside standard utilities.
All synthetic data generation is driven by **three provider types**:

- **`FakerSyntheticDataProvider`** — wraps [Faker](https://faker.readthedocs.io/) for realistic personal data.
- **`RandomChoiceDataProvider`** — discrete sampling (gender, country, account type, …).
- **`UniformDistributionDataProvider`** — continuous uniform sampling (tower perturbation, amounts, …).


In [1]:
import sys
sys.path.insert(0, "..")

import typing
import yaml
import numpy as np
import pandas as pd

from data_generation.attributes import (
    get_per_participant_attributes,
    get_payment_core_attributes,
    get_payment_amount_attributes,
)
from data_generation.dataset import topological_sort, generate
from data_generation.synthetic_data_provider import (
    FakerSyntheticDataProvider,
    RandomChoiceDataProvider,
    UniformDistributionDataProvider,
)
from data_generation.rng.uniform_distribution import UniformDistributionSamplingConfig
from data_generation.static_data import country_static_data
from data_generation.anomaly_transformers import (
    add_fraud_columns,
    inject_all,
    apply_fraud_with_probability,
    Type1Config,
    Type2Config,
)

## Load Site Configuration

Dataset generation is parameterised via per-site YAML files in `config/{site_name}.yml`.
Each site config contains:

- **`anomaly_generation_config.field`** — distribution parameters for tower perturbation,
  normal/anomalous payment amounts (lognormal), etc.
- **`dataset_generation_config`** — a list of dataset specs, each with its own anomaly
  rule stack, row count, fraud probability, and output label.

This design allows each FL participant (bank/site) to have a *different* data distribution,
which is exactly the scenario federated learning is designed to handle.

In [2]:
SITE_NAME = "site1"

with open(f"../config/{SITE_NAME}.yml") as f:
    site_config: dict = yaml.safe_load(f)

fields = site_config["anomaly_generation_config"]["field"]
dataset_configs = site_config["dataset_generation_config"]

# Extract tower perturbation configs (uniform distributions)
tower_lat_cfg = UniformDistributionSamplingConfig(
    low=fields["tower_lat"]["distributions"][0]["low"],
    high=fields["tower_lat"]["distributions"][0]["high"],
)
tower_lon_cfg = UniformDistributionSamplingConfig(
    low=fields["tower_long"]["distributions"][0]["low"],
    high=fields["tower_long"]["distributions"][0]["high"],
)

# Extract lognormal amount configs from site parameters
personal_amount_cfg = fields["normal_personal_acc_amount"]["distributions"][0]
business_amount_cfg = fields["normal_business_acc_amount"]["distributions"][0]

# Use the first dataset config entry for exploration
ds_cfg = dataset_configs[0]

print(f"Site: {SITE_NAME}")
print(f"Dataset configs: {len(dataset_configs)} entries")
print(f"Exploring: label={ds_cfg['fname_label']!r}, rows={ds_cfg['max_num_rows']}, rules={ds_cfg['fraud_insertion_rule_stack']}")
print(f"Tower lat perturbation: [{tower_lat_cfg.low}, {tower_lat_cfg.high}]")
print(f"Tower lon perturbation: [{tower_lon_cfg.low}, {tower_lon_cfg.high}]")
print(f"Normal personal amount: lognormal(mean={personal_amount_cfg['desired_mean']}, σ={personal_amount_cfg['sigma']})")
print(f"Normal business amount: lognormal(mean={business_amount_cfg['desired_mean']}, σ={business_amount_cfg['sigma']})")

Site: site1
Dataset configs: 6 entries
Exploring: label='train', rows=100000, rules=['type1', 'type2', 'type3']
Tower lat perturbation: [-0.75, 1.25]
Tower lon perturbation: [-1.5, 2]
Normal personal amount: lognormal(mean=20000, σ=7500)
Normal business amount: lognormal(mean=80000, σ=10000)


## Static Data — Countries, Currencies & Exchange Rates

Reference data is loaded (or generated once and cached) via `country_static_data.load_static_data()`.
This includes:

- **Country → currency mapping** derived from `babel.numbers.get_territory_currencies()` for a fixed set
  of 9 countries: US, TR, AT, IE, PL, PT, GB, FR, IN.
- **Pairwise exchange rates** fetched from the ECB via `CurrencyConverter` (snapshot date: 2019-01-01)
  and cached to `~/.cache/fsi_static/currency_exchange_rates.csv`.

> The exchange rate CSV is generated once on first run.  Set `force_rates_from_api=True` to re-fetch.


In [3]:
static_data = country_static_data.load_static_data("~/.cache/fsi_static")

print("Country → Currency mapping:")
display(static_data.country_currency_map)

print("\nExchange rates (first 10 rows):")
display(static_data.currency_exchange_rates.head(10))


Country → Currency mapping:


,country,currency
0,US,USD
1,TR,TRY
2,AT,EUR
3,IE,EUR
4,PL,PLN
5,PT,EUR
6,GB,GBP
7,FR,EUR
8,IN,INR



Exchange rates (first 10 rows):


,CCY1,CCY2,RATE
0,USD,USD,1.0000
1,USD,TRY,5.3275
2,USD,EUR,0.8754
3,USD,EUR,0.8754
4,USD,PLN,3.7632
5,USD,EUR,0.8754
6,USD,GBP,0.7862
7,USD,EUR,0.8754
8,USD,INR,69.9065
9,TRY,USD,0.1877


## Data Providers

We instantiate one provider per type, all sharing the same **seed** for reproducibility.
The seed ensures that re-running the notebook produces identical datasets.

| Provider | Backs | Example attributes |
|----------|-------|--------------------|
| `FakerSyntheticDataProvider` | Faker | username, email, DOB, geo-coordinates |
| `RandomChoiceDataProvider` | `np.random.default_rng` | gender, country, account type, payment status |
| `UniformDistributionDataProvider` | `np.random.default_rng` | tower perturbation, activity events, amounts |


In [4]:
SEED = 42

faker_provider = FakerSyntheticDataProvider(seed=SEED)
random_choice_provider = RandomChoiceDataProvider(seed=SEED)
uniform_provider = UniformDistributionDataProvider(seed=SEED)

print(f"Providers created with seed={SEED}")


Providers created with seed=42


## Building the Dependency Graph

The dependency graph is the heart of the generation pipeline.  Each **attribute** (column or column-group)
is mapped to the list of columns it depends on.  The graph is built in three layers:

1. **Per-participant attributes** — duplicated for DEBITOR and CREDITOR prefixes (personal info, address, DOB, geo, timestamps, …).
2. **Payment core attributes** — payment ID, initiation/update timestamps, status.
3. **Payment amount attributes** — exchange rates and debitor/creditor amounts.

Attributes with an empty dependency list are **independent** and can be generated first.
Dependent attributes (e.g. currency depends on country, tower coordinates depend on geo-coordinates)
are generated after their prerequisites.


In [5]:
PREFIXES = ("DEBITOR", "CREDITOR")

# Build the full graph incrementally
graph = get_per_participant_attributes(PREFIXES)
get_payment_core_attributes(PREFIXES, graph)
get_payment_amount_attributes(PREFIXES, graph)

n_independent = sum(1 for deps in graph.values() if not deps)
n_dependent = sum(1 for deps in graph.values() if deps)

print(f"Total attributes: {len(graph)}")
print(f"  Independent (no dependencies): {n_independent}")
print(f"  Dependent: {n_dependent}")


Total attributes: 54
  Independent (no dependencies): 38
  Dependent: 16


## Topological Sort — Generation Order

`topological_sort()` implements **Kahn's algorithm** to determine a valid generation order.
It builds a reverse index (column name → producing attribute), computes in-degrees, and
BFS-walks to yield a dependency-respecting sequence.

- All independent attributes come first (in graph-insertion order).
- Each dependent attribute appears *after* every attribute that produces its prerequisite columns.
- A `ValueError` is raised if the graph contains a cycle.


In [6]:
order = topological_sort(graph)

print("Generation order:")
for i, attr in enumerate(order, 1):
    deps = graph[attr]
    suffix = f"  ← {deps}" if deps else ""
    print(f"  {i:>2}. {attr.names}{suffix}")


Generation order:
   1. ('DEBITOR_USERNAME',)
   2. ('DEBITOR_FIRST_NAME',)
   3. ('DEBITOR_LAST_NAME',)
   4. ('DEBITOR_EMAIL_ADDRESS',)
   5. ('DEBITOR_PHONE_NUMBER',)
   6. ('DEBITOR_GENDER',)
   7. ('DEBITOR_ACCOUNT_NUMBER',)
   8. ('DEBITOR_BIC_CODE',)
   9. ('DEBITOR_IP_ADDRESS',)
  10. ('DEBITOR_COMMENT',)
  11. ('DEBITOR_ACCOUNT_TYPE',)
  12. ('DEBITOR_ADDR_BUILDING',)
  13. ('DEBITOR_ADDR_STREET',)
  14. ('DEBITOR_ADDR_CITY',)
  15. ('DEBITOR_ADDR_STATE',)
  16. ('DEBITOR_ADDR_ZIPCODE',)
  17. ('DEBITOR_ADDR_COUNTRY',)
  18. ('DEBITOR_BIRTH_YEAR', 'DEBITOR_BIRTH_MONTH', 'DEBITOR_BIRTH_DAY')
  19. ('CREDITOR_USERNAME',)
  20. ('CREDITOR_FIRST_NAME',)
  21. ('CREDITOR_LAST_NAME',)
  22. ('CREDITOR_EMAIL_ADDRESS',)
  23. ('CREDITOR_PHONE_NUMBER',)
  24. ('CREDITOR_GENDER',)
  25. ('CREDITOR_ACCOUNT_NUMBER',)
  26. ('CREDITOR_BIC_CODE',)
  27. ('CREDITOR_IP_ADDRESS',)
  28. ('CREDITOR_COMMENT',)
  29. ('CREDITOR_ACCOUNT_TYPE',)
  30. ('CREDITOR_ADDR_BUILDING',)
  31. ('CREDITOR_AD

## Generating the Normal Payment Dataset

The `generate()` function:

1. Topologically sorts the graph.
2. Creates an empty `pd.DataFrame` with `n_rows` rows.
3. Iterates through attributes in order, calling each attribute's `emit()` method.
4. For single-column attributes, assigns the returned `pd.Series` to the DataFrame.
5. For multi-column attributes, maps each result column to its named attribute.

We map each attribute to its provider by inspecting the `provider` type annotation on the
underlying helper function—this keeps wiring automatic and DRY.


In [7]:
N_ROWS = ds_cfg["max_num_rows"]

# Auto-map each attribute → provider via type annotations
provider_map = {
    FakerSyntheticDataProvider: faker_provider,
    RandomChoiceDataProvider: random_choice_provider,
    UniformDistributionDataProvider: uniform_provider,
}
providers = {}
for attr in graph:
    hints = typing.get_type_hints(attr.attribute_data_provider)
    providers[attr] = provider_map[hints["provider"]]

# Generate with lognormal amount distributions from site config
df = generate(
    graph,
    providers,
    N_ROWS,
    country_static_data=static_data,
    uniform_dist_config_lat=tower_lat_cfg,
    uniform_dist_config_lon=tower_lon_cfg,
    lognormal_personal_amount_config=personal_amount_cfg,
    lognormal_business_amount_config=business_amount_cfg,
)

print(f"Generated {len(df)} rows × {len(df.columns)} columns")

Generated 100000 rows × 64 columns


## Exploring the Generated Data

Let's inspect the generated DataFrame — column names, data types, and a sample of rows.


In [8]:
print("Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

print(f"\nShape: {df.shape}")
print(f"\nData types:\n{df.dtypes.value_counts()}")


Columns:
   1. DEBITOR_USERNAME
   2. DEBITOR_FIRST_NAME
   3. DEBITOR_LAST_NAME
   4. DEBITOR_EMAIL_ADDRESS
   5. DEBITOR_PHONE_NUMBER
   6. DEBITOR_GENDER
   7. DEBITOR_ACCOUNT_NUMBER
   8. DEBITOR_BIC_CODE
   9. DEBITOR_IP_ADDRESS
  10. DEBITOR_COMMENT
  11. DEBITOR_ACCOUNT_TYPE
  12. DEBITOR_ADDR_BUILDING
  13. DEBITOR_ADDR_STREET
  14. DEBITOR_ADDR_CITY
  15. DEBITOR_ADDR_STATE
  16. DEBITOR_ADDR_ZIPCODE
  17. DEBITOR_ADDR_COUNTRY
  18. DEBITOR_BIRTH_YEAR
  19. DEBITOR_BIRTH_MONTH
  20. DEBITOR_BIRTH_DAY
  21. CREDITOR_USERNAME
  22. CREDITOR_FIRST_NAME
  23. CREDITOR_LAST_NAME
  24. CREDITOR_EMAIL_ADDRESS
  25. CREDITOR_PHONE_NUMBER
  26. CREDITOR_GENDER
  27. CREDITOR_ACCOUNT_NUMBER
  28. CREDITOR_BIC_CODE
  29. CREDITOR_IP_ADDRESS
  30. CREDITOR_COMMENT
  31. CREDITOR_ACCOUNT_TYPE
  32. CREDITOR_ADDR_BUILDING
  33. CREDITOR_ADDR_STREET
  34. CREDITOR_ADDR_CITY
  35. CREDITOR_ADDR_STATE
  36. CREDITOR_ADDR_ZIPCODE
  37. CREDITOR_ADDR_COUNTRY
  38. CREDITOR_BIRTH_YEAR
  39. CREDI

In [9]:
df.head(5)


,DEBITOR_USERNAME,DEBITOR_FIRST_NAME,DEBITOR_LAST_NAME,DEBITOR_EMAIL_ADDRESS,DEBITOR_PHONE_NUMBER,DEBITOR_GENDER,DEBITOR_ACCOUNT_NUMBER,DEBITOR_BIC_CODE,DEBITOR_IP_ADDRESS,DEBITOR_COMMENT,...,DEBITOR_ACCOUNT_LAST_ACTIVITY_TIMESTAMP,DEBITOR_CCY_CREDITOR_CCY_RATE,CREDITOR_CCY_DEBITOR_CCY_RATE,CREDITOR_TOWER_LATITUDE,CREDITOR_TOWER_LONGITUDE,CREDITOR_ACCOUNT_LAST_ACTIVITY_TIMESTAMP,DEBITOR_AMOUNT,CREDITOR_AMOUNT,PAYMENT_INIT_TIMESTAMP,PAYMENT_LAST_UPDATE_TIMESTAMP
0,yildirimsatrettin,Mark,Smith,jennifermullen@example.com,+90(368)929-0800x022,M,GB80EUGZ39924669902089,032241073,38.165.29.216,Quisquam maiores temporibus quidem quis.,...,1.774249e+09,0.0125,79.8577,41.254213,-8.569525,1.774570e+09,90149.45,1126.87,1.775174e+09,1.775174e+09
1,muhammetyaman,Christopher,Sakarya,sebattincamurcuoglu@example.org,577-609-8473,O,TR196430201150914099349395,035327800,172.24.160.99,Line lawyer above item simple.,...,1.774115e+09,0.0143,69.9065,33.555921,-95.073900,1.774229e+09,108914.58,1557.48,1.774833e+09,1.774833e+09
2,tarhanhasgul,Yahşikan,Davies,kpotter@example.net,001-676-497-9362x528,F,GB86KDQN14686335704637,021905757,141.200.86.127,Simply risk need cost one.,...,1.774327e+09,0.0125,79.8577,49.214321,4.079219,1.774328e+09,21131.94,264.15,1.774933e+09,1.774933e+09
3,johnsonjoshua,Vincent,Owen,susanstephens@example.org,899-758-6087,F,GB33IOYQ02908449392795,015690133,139.144.190.237,Fine indeed understand.,...,1.774453e+09,1.0000,1.0000,10.825508,78.044301,1.774356e+09,40187.88,40187.88,1.775058e+09,1.775058e+09
4,johnsonjoshua,Courtney,Demir,musure77@example.net,001-783-390-6387,F,TR274005446480143879694025,020827056,200.103.253.33,Totam hic ducimus reiciendis non exercitationem.,...,1.774475e+09,1.0000,1.0000,53.478997,-6.731298,1.774252e+09,9025.87,9025.87,1.775079e+09,1.775079e+09


In [10]:
# Quick sanity checks on key columns
print("=== Debitor Countries ===")
print(df["DEBITOR_ADDR_COUNTRY"].value_counts())

print("\n=== Debitor Account Types ===")
print(df["DEBITOR_ACCOUNT_TYPE"].value_counts())

print("\n=== Payment Status ===")
print(df["PAYMENT_STATUS"].value_counts())

print("\n=== Amount statistics ===")
print(df[["DEBITOR_AMOUNT", "CREDITOR_AMOUNT"]].describe())


=== Debitor Countries ===
DEBITOR_ADDR_COUNTRY
IE    11259
TR    11210
GB    11119
PT    11106
PL    11096
US    11082
IN    11071
AT    11050
FR    11007
Name: count, dtype: int64

=== Debitor Account Types ===
DEBITOR_ACCOUNT_TYPE
CHECKING    33463
BUSINESS    33337
SAVINGS     33200
Name: count, dtype: int64

=== Payment Status ===
PAYMENT_STATUS
PENDING       20188
CANCELLED     20146
COMPLETED     19923
PROCESSING    19876
FAILED        19867
Name: count, dtype: int64

=== Amount statistics ===
       DEBITOR_AMOUNT  CREDITOR_AMOUNT
count   100000.000000     1.000000e+05
mean     39988.148316     3.003526e+05
std      29527.035130     1.042298e+06
min       3464.670000     5.042000e+01
25%      16655.472500     1.212564e+04
50%      23889.290000     2.414143e+04
75%      73080.272500     8.731436e+04
max     137585.810000     9.922552e+06


## Anomaly Generation

The approach to generating anomalous data is **perturbative**: we treat the base dataset as
"normal" (non-fraudulent) transactions, then apply rule-based mutations to inject anomalies.

Each anomaly type targets different dimensions of the data:

| Type | Module | Description |
|------|--------|-------------|
| 1 | `anomaly_transformers.type1` | Tower coordinates are far from physical location |
| 2 | `anomaly_transformers.type2` | Account created minutes before a large transaction |
| 3 | `anomaly_transformers.type3` | Last activity was months before the current transaction |
| 4 | `anomaly_transformers.type4` | Unusually high event counts in the past 30 days |

These rules can be applied **independently** or **combined** with controlled overlap to create
complex multi-dimensional anomaly scenarios.

All anomaly transformers operate on DataFrame slices in **bulk** using vectorised numpy operations
rather than row-by-row `apply()`. They are configured from `parameters.yml` via dataclass configs
(`Type1Config`, `Type2Config`). Types 3 and 4 have no site-specific parameters.

In [11]:
# Build anomaly configs from site parameters
type1_cfg = Type1Config.from_site_fields(fields)
type2_cfg = Type2Config.from_site_fields(fields)

anomaly_configs = {
    "type1": {"config": type1_cfg},
    "type2": {"config": type2_cfg},
    # type3 and type4 have no site-specific config
}

print(f"Type 1 config: NorE=[{type1_cfg.nor_e_low}, {type1_cfg.nor_e_high}], SorW=[{type1_cfg.sor_w_low}, {type1_cfg.sor_w_high}]")
print(f"Type 2 config: personal(mean={type2_cfg.personal_mean}, σ={type2_cfg.personal_sigma}), business(mean={type2_cfg.business_mean}, σ={type2_cfg.business_sigma})")
print(f"Anomaly types for {SITE_NAME}: {ds_cfg['fraud_insertion_rule_stack']}")
print(f"Fraud overlap frac: {ds_cfg.get('fraud_overlap_frac', 0.1)}")
print(f"Apply probability: {ds_cfg.get('apply_probability', 0.9)}")

Type 1 config: NorE=[-4.5, -4.5], SorW=[-4.5, -4.5]
Type 2 config: personal(mean=75000, σ=5000), business(mean=240000, σ=15000)
Anomaly types for site1: ['type1', 'type2', 'type3']
Fraud overlap frac: 0.1
Apply probability: 0.9


## Inject Anomalies

Using `inject_all()` from the `anomaly_transformers` module to apply each anomaly type
configured for this site. The framework:

1. Adds fraud-tracking columns (`FRAUD_FLAG`, `TYPE{1-4}_ANOMALY`).
2. For each anomaly type, samples a random fraction of rows (0.1–1 %) and applies the
   vectorised transformer in bulk.
3. Controlled overlap (`fraud_overlap_frac=0.1`) allows ~10 % of already-flagged rows to
   receive additional anomaly types, creating multi-dimensional fraud signatures.

In [12]:
# Work on a copy so we don't mutate the original
test_df = df.copy()

# Shuffle rows for randomness
test_df = test_df.sample(frac=1, replace=False, random_state=SEED).reset_index(drop=True)

# Add fraud-tracking columns
add_fraud_columns(test_df)

# Inject all anomaly types from the first dataset config entry
test_df = inject_all(
    test_df,
    anomaly_types=ds_cfg["fraud_insertion_rule_stack"],
    configs=anomaly_configs,
    fraud_overlap_frac=ds_cfg.get("fraud_overlap_frac", 0.1),
    seed=SEED,
)

n_fraud = (test_df["FRAUD_FLAG"] == 1).sum()
print(f"Fraud rows before thinning: {n_fraud} / {len(test_df)} ({n_fraud/len(test_df)*100:.2f}%)")
for t in ["TYPE1_ANOMALY", "TYPE2_ANOMALY", "TYPE3_ANOMALY", "TYPE4_ANOMALY"]:
    count = test_df[t].sum()
    if count > 0:
        print(f"  {t}: {count}")

# Apply fraud probability thinning
apply_prob = ds_cfg.get("apply_probability", 0.9)
apply_fraud_with_probability(test_df, prob=apply_prob, random_state=SEED)
n_fraud_after = (test_df["FRAUD_FLAG"] == 1).sum()
print(f"\nAfter probability thinning (prob={apply_prob}): {n_fraud_after} fraud rows")

Fraud rows before thinning: 1150 / 100000 (1.15%)
  TYPE1_ANOMALY: 410
  TYPE2_ANOMALY: 444
  TYPE3_ANOMALY: 379

After probability thinning (prob=0.9): 1035 fraud rows


In [13]:
# Inspect anomalous features
anomaly_cols = [
    "FRAUD_FLAG", "TYPE1_ANOMALY", "TYPE2_ANOMALY", "TYPE3_ANOMALY", "TYPE4_ANOMALY",
    "DEBITOR_TOWER_LATITUDE", "DEBITOR_TOWER_LONGITUDE",
    "DEBITOR_ACCOUNT_CREATE_TIMESTAMP", "DEBITOR_AMOUNT",
    "DEBITOR_ACCOUNT_LAST_ACTIVITY_TIMESTAMP",
    "DEBITOR_ACCOUNT_ACTIVITY_EVENTS_PAST_30D",
]
test_df[test_df["FRAUD_FLAG"] == 1][anomaly_cols].head(10)


,FRAUD_FLAG,TYPE1_ANOMALY,TYPE2_ANOMALY,TYPE3_ANOMALY,TYPE4_ANOMALY,DEBITOR_TOWER_LATITUDE,DEBITOR_TOWER_LONGITUDE,DEBITOR_ACCOUNT_CREATE_TIMESTAMP,DEBITOR_AMOUNT,DEBITOR_ACCOUNT_LAST_ACTIVITY_TIMESTAMP,DEBITOR_ACCOUNT_ACTIVITY_EVENTS_PAST_30D
42,1,False,False,True,False,28.226431,79.494978,1.202583e+09,39304.56,1.762189e+09,92
64,1,False,False,True,False,42.419366,1.383511,-1.208181e+07,22446.31,1.762264e+09,292
263,1,False,True,False,False,49.066019,2.165166,1.774927e+09,255104.19,1.774326e+09,1476383
392,1,True,True,False,False,74.421039,-23.418207,1.774835e+09,257033.61,1.774121e+09,638455
592,1,True,False,False,False,23.507131,151.763366,3.443094e+08,22221.60,1.774142e+09,26
657,1,False,False,True,False,37.094994,30.947816,1.505747e+09,18212.69,1.759320e+09,132
659,1,False,True,False,False,38.128686,-24.076117,1.774694e+09,68237.94,1.774066e+09,145
1281,1,False,True,False,False,54.775123,-7.874324,1.775001e+09,240175.32,1.774396e+09,1163917
1379,1,True,False,False,False,17.974978,-100.037584,1.552092e+09,33788.62,1.774195e+09,10
1440,1,True,False,False,False,-4.519364,-105.330810,1.155529e+09,78855.30,1.774248e+09,975295


## Summary

We have successfully:

1. **Loaded site configuration** from `config/{site_name}.yml` using the new per-site config structure
   (`anomaly_generation_config` + `dataset_generation_config`).
2. **Built a dependency graph** of 54 attributes across debitor, creditor, and payment dimensions.
3. **Topologically sorted** the graph to determine a valid generation order.
4. **Generated a full DataFrame** of normal payment transactions with account-type-aware
   lognormal amount distributions loaded from site config.
5. **Injected rule-based anomalies** using `data_generation.anomaly_transformers`:
   - All four anomaly types implemented as vectorised bulk `apply()` functions.
   - Configs loaded from site YAML via `Type1Config.from_site_fields()` / `Type2Config.from_site_fields()`.
   - Injection framework handles overlap control.
6. **Applied fraud probability thinning** via `apply_fraud_with_probability()` to create
   realistic hard negatives — rows with anomalous features but `FRAUD_FLAG=0`.

### CLI Tool

The full bulk generation pipeline is available via `main.py`:

```bash
# Generate datasets for specific sites
uv run main.py -s site1 -s siteB -o output/ -S 42

# Generate for all sites
uv run main.py -o output/
```

Each site config drives multiple dataset specs (train, scaling, eval) with independent
anomaly rule stacks, row counts, and fraud probability thinning.